In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from torch.amp import autocast, GradScaler
from torch import device, cuda, nn , no_grad, tensor,load,save, sigmoid , cat , float32
from torch.utils.data import Dataset,DataLoader
from torchvision.utils import make_grid
from torchvision.models import resnet50,vit_b_16,swin_v2_s,inception_v3,vgg19,ResNet50_Weights,ViT_B_16_Weights,Swin_V2_S_Weights,Inception_V3_Weights,VGG19_Weights
import torchvision.transforms as transforms
from torch.optim import AdamW,lr_scheduler
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score,f1_score, precision_score, recall_score,multilabel_confusion_matrix
import seaborn as sns
import time
from warnings import filterwarnings


filterwarnings("ignore")

%matplotlib inline

dev = device("cuda" if cuda.is_available() else "cpu")
dev

In [ ]:
checkpoint = load(r"data_essentials.pth",map_location=dev,weights_only=False)
checkpoint.keys()

In [ ]:
print(cuda.is_available())     
print(cuda.get_device_name(0))

In [ ]:
SAVED_MODELS_FOLDER = r"C:\Users\ibrah.HIMA\OneDrive\Desktop\Full AI\My Reserch Papers\chest-xray-multilabel-classification\saved_models"

PATH = r"D:\Data"

# Collect paths for all 12 image subfolders
# Each folder follows the pattern: images_00X/images/
IMG_FOLDERS = [
    os.path.join(PATH, x + "/images")
    for x in os.listdir(PATH)
    if x.startswith("images_")
]

NUM_OF_CLASSES = checkpoint["num_of_classes"]

ALL_DISEASES = checkpoint["all_diseases"]

BATCH_SIZE = 128
MEAN_NORM = [0.485, 0.456, 0.406]
STD_NORM = [0.229, 0.224, 0.225]

POS_WEIGHTS = checkpoint["pos_weights"]

EPOCHS = 30
PATIENCE = 5
BEST_VAL_AUC = 0.0

LR = 1e-4

train_df = checkpoint["train_df"]
val_df = checkpoint["val_df"]
test_df = checkpoint["test_df"]

In [ ]:
def get_transforms(size=224 , In_Train=True):
    if In_Train:
        return transforms.Compose([transforms.Resize((size,size)),
                                   transforms.RandomRotation(10),
                                   transforms.RandomHorizontalFlip(),
                                   
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean=MEAN_NORM,
                                                        std=STD_NORM)])
    else:
        return transforms.Compose([transforms.Resize((size,size)),
                                   transforms.ToTensor(),
                                   transforms.Normalize(mean=MEAN_NORM,
                                                        std=STD_NORM)])

In [ ]:
class ChestXRayDataset(Dataset):
    def __init__(self,df,img_folders,transforms=None):
        self.df = df.reset_index(drop=True)
        self.img_folders = img_folders
        self.transforms = transforms
        self.labels = self._encode_labels()

    def _encode_labels(self):
        encode = []

        for label in self.df["Finding Labels"]:
            diseases_in_img = label.split("|")

            vector = [0.0] * NUM_OF_CLASSES
            for disease in diseases_in_img:
                if disease in ALL_DISEASES:
                    idx = ALL_DISEASES.index(disease)
                    vector[idx] = 1.0
            encode.append(vector)
        return encode
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        img_name = self.df.loc[index,"Image Index"]

        img = None
        for folder in IMG_FOLDERS:
            img_path = os.path.join(folder,img_name)
            if os.path.exists(img_path):
                img = Image.open(img_path).convert("RGB")
                break
        
        if self.transforms:
            img = self.transforms(img)

        label = tensor(self.labels[index],dtype=float32)
        return img,label


train_dataset = ChestXRayDataset(train_df,IMG_FOLDERS,transforms=get_transforms())
val_dataset = ChestXRayDataset(val_df,IMG_FOLDERS,get_transforms(In_Train=False))
test_dataset = ChestXRayDataset(test_df,IMG_FOLDERS,get_transforms(In_Train=False))

inc_train_dataset = ChestXRayDataset(train_df,IMG_FOLDERS,get_transforms(299))
inc_val_dataset = ChestXRayDataset(val_df,IMG_FOLDERS,get_transforms(299,In_Train=False))
inc_test_dataset = ChestXRayDataset(test_df,IMG_FOLDERS,get_transforms(299,In_Train=False))

swin_train_dataset = ChestXRayDataset(train_df,IMG_FOLDERS,get_transforms(256))
swin_val_dataset = ChestXRayDataset(val_df,IMG_FOLDERS,get_transforms(256,In_Train=False))
swin_test_dataset = ChestXRayDataset(test_df,IMG_FOLDERS,get_transforms(256,In_Train=False))

In [ ]:
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_dl = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_dl = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

inc_train_dl = DataLoader(inc_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
inc_val_dl = DataLoader(inc_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
inc_test_dl = DataLoader(inc_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

swin_train_dl = DataLoader(swin_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
swin_val_dl = DataLoader(swin_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
swin_test_dl = DataLoader(swin_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
def denormalize(data):

    
    mean = tensor(MEAN_NORM).to(data.device)
    std = tensor(STD_NORM).to(data.device)

    
    if data.ndimension() == 3:
        
        res = data * std[:, None, None] + mean[:, None, None]
        
    
    elif data.ndimension() == 4:
       
        res = data * std[None, :, None, None] + mean[None, :, None, None]
    
    else:
        return data 

    return res.clamp(0, 1)

In [ ]:
show_train_dl = DataLoader(train_dataset, batch_size=16, shuffle=True)
show_val_dl = DataLoader(val_dataset, batch_size=16, shuffle=False)
show_test_dl = DataLoader(test_dataset, batch_size=16, shuffle=False)

def Show_Sample(dl):
    _,ax = plt.subplots(figsize=(12,10))
    imgs,labels = next(iter(dl))
    imgs = denormalize(imgs)
    ax.set_yticks([])
    ax.set_xticks([])
    grid = make_grid(imgs, 4).permute(1, 2, 0).numpy()


    ax.imshow(grid)
    

Show_Sample(show_train_dl)

In [ ]:
Show_Sample(show_val_dl)

In [ ]:
Show_Sample(show_test_dl)

In [ ]:
def Show_Curves(model_history):
    train_loss_hist = [x["train_loss"] for x in model_history]
    val_loss_hist   = [x["val_loss" ] for x in model_history]
    train_auc_hist  = [x["train_auc"  ] for x in model_history]
    val_auc_hist    = [x["val_auc"  ] for x in model_history]

    _, ax = plt.subplots(1, 2, figsize=(14, 5))

    # ── Loss Curve ──────────────────────────────────────────────────────────
    ax[0].plot(train_loss_hist,     label="Train",      linewidth=2)
    ax[0].plot(val_loss_hist, label="Validation",  linewidth=2, linestyle="--")
    ax[0].set_title("Loss Over Epochs", fontsize=13, fontweight="bold")
    ax[0].set_ylabel("Loss")
    ax[0].set_xlabel("Epoch")
    ax[0].legend()
    ax[0].grid(alpha=0.3)

    # ── AUC Curve ───────────────────────────────────────────────────────
    ax[1].plot(train_auc_hist,     label="Train",      linewidth=2)
    ax[1].plot(val_auc_hist, label="Validation",  linewidth=2, linestyle="--")
    ax[1].set_title("AUC Over Epochs", fontsize=13, fontweight="bold")
    ax[1].set_ylabel("Accuracy")
    ax[1].set_xlabel("Epoch")
    ax[1].legend()
    ax[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join('AUC_LOSS_Curves.png'), dpi=300,bbox_inches='tight')
    plt.show()

In [ ]:
def Show_MCM(labels,preds_binary):

    mcm = multilabel_confusion_matrix(labels, preds_binary)

    fig, axes = plt.subplots(3, 5, figsize=(20, 12))
    axes = axes.flatten()

    for i, (matrix, disease) in enumerate(zip(mcm, ALL_DISEASES)):
        sns.heatmap(matrix, 
                    annot=True, 
                    fmt='d',
                    cmap='Blues',
                    ax=axes[i],
                    xticklabels=['Pred 0', 'Pred 1'],
                    yticklabels=['True 0', 'True 1'])
        axes[i].set_title(disease, fontsize=10)

    fig.delaxes(axes[14])
    plt.suptitle('Confusion Matrix per Disease', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join('diseases_confusion_matrices.png'), dpi=300,bbox_inches='tight')
    plt.show()

In [ ]:
def model_predict(model,test_dl):
    # all_imgs = []
    all_preds = []
    all_labels = []

    model.eval()

    with no_grad():
        for batch in test_dl:
            imgs,labels = batch
            imgs = imgs.to(dev)
            labels = labels.to(dev)
            
            with autocast(device_type='cuda'):
                outputs = model(imgs)
            
            probs = sigmoid(outputs).float()
            
            # all_imgs.extend(imgs.cpu())
            all_preds.append(probs.detach().cpu())
            all_labels.append(labels.detach().cpu())
        
    all_preds = cat(all_preds).numpy()
    all_labels = cat(all_labels).numpy()
    
    best_thresholds = []
    
    for i in range(14):
        best_t = 0.5
        best_f1 = 0
    
        for t in np.arange(0.1, 0.9, 0.05):
            preds = (all_preds[:, i] > t).astype(int)
            
            f1 = f1_score(all_labels[:, i], preds)
            
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
    
        best_thresholds.append(best_t)
    print(best_thresholds)

    return (all_labels,all_preds,best_thresholds)

In [ ]:
def replace_classifier(model,name):
    """
    Replace the final classification layer of any supported model
    with a new linear layer matching NUM_OF_CLASSES (14 diseases).
    """

    if "VGG" in name:
        model.classifier[6] = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.classifier[6].in_features, NUM_OF_CLASSES)
        )

    elif "Inception" in name:
        model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.fc.in_features, NUM_OF_CLASSES))
        if model.AuxLogits is not None:
            model.AuxLogits.fc = nn.Sequential(
                nn.Dropout(p=0.3),
                nn.Linear(model.AuxLogits.fc.in_features, NUM_OF_CLASSES)
            )
                
    elif "ResNet" in name:
        model.fc = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.fc.in_features, NUM_OF_CLASSES)
        )
    
    elif "VisionTransformer" in name:
        model.heads = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.heads.head.in_features, NUM_OF_CLASSES)
        )

    elif "SwinTransformer" in name:
        model.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(model.head.in_features, NUM_OF_CLASSES)
        )

    return model

In [ ]:
def fit_predict(model_name, model_weights, all_splitted_data,
                show_logs=True, show_results=True):

    model = model_name(model_weights.DEFAULT)
    name = model.__class__.__name__
    model = replace_classifier(model,name)
    
    
    
        
    model = model.to(dev)
    if cuda.device_count() > 1:
        model = nn.DataParallel(model)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHTS,reduction='mean')
    opt = AdamW(model.parameters(), lr=LR,weight_decay=1e-2)
    scheduler = lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=EPOCHS,
        eta_min=1e-6    
    )
    
    best_val_auc = 0.0
    counter = 0

    save_best_model_path = os.path.join(
        SAVED_MODELS_FOLDER, f'best_{name}_model.pth'
    )

    if show_logs:
        print(f"{name} Is Running")

    history = []
    scaler = GradScaler()
    start = time.time()
    for epoch in range(EPOCHS):
        # ── TRAIN ──────────────────────────────────────────
        all_train_loss   = []
        all_train_preds  = []
        all_train_labels = []
        model.train()
        
        for batch in tqdm(all_splitted_data[0],
                          desc=f"Epoch {epoch+1}/{EPOCHS}" if show_logs else None):
            imgs, labels = batch
            imgs   = imgs.to(dev, non_blocking=True)  
            labels = labels.to(dev, non_blocking=True)

            opt.zero_grad()                            

            with autocast(device_type='cuda'):
                outputs, aux_outputs = model(imgs)
                loss1 = criterion(outputs, labels)
                loss2 = criterion(aux_outputs, labels)
                
                loss = loss1 + 0.4 * loss2

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            all_train_loss.append(loss.item())
            all_train_preds.append(sigmoid(outputs).detach().cpu())
            all_train_labels.append(labels.detach().cpu())

        scheduler.step()
        all_train_preds  = cat(all_train_preds).numpy()
        all_train_labels = cat(all_train_labels).numpy()

        valid_classes = [i for i in range(NUM_OF_CLASSES)
                         if all_train_labels[:, i].sum() > 0]
        
        train_auc  = roc_auc_score(all_train_labels[:, valid_classes],
                                   all_train_preds [:, valid_classes], average="macro")
        
        train_loss = sum(all_train_loss) / len(all_train_loss)

        # ── EVAL ───────────────────────────────────────────
        all_val_loss   = []
        all_val_preds  = []
        all_val_labels = []
        model.eval()

        with no_grad():
            for batch in all_splitted_data[1]:
                imgs, labels = batch
                imgs   = imgs.to(dev, non_blocking=True)
                labels = labels.to(dev, non_blocking=True)
                
                with autocast(device_type='cuda'):
                    outputs = model(imgs)
                    loss    = criterion(outputs, labels) 

                all_val_loss.append(loss.item())
                all_val_preds.append(sigmoid(outputs).detach().cpu())
                all_val_labels.append(labels.detach().cpu())

        val_loss = sum(all_val_loss) / len(all_val_loss)
        all_val_preds  = cat(all_val_preds).numpy()
        all_val_labels = cat(all_val_labels).numpy()

        valid_classes = [i for i in range(NUM_OF_CLASSES)
                         if all_val_labels[:, i].sum() > 0]
        val_auc = roc_auc_score(all_val_labels[:, valid_classes],
                                all_val_preds[:, valid_classes], average="macro")

        if show_logs:
            print(f"Train Loss: {train_loss:.4f} | "
                  f"Val Loss: {val_loss:.4f} | "
                  f"Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}")

        history.append({
            "train_loss": train_loss, "val_loss": val_loss,
            "train_auc":  train_auc,  "val_auc":  val_auc,
        })

        # ── EARLY STOPPING ────────────────────────────────
        if val_auc > best_val_auc:          
            best_val_auc = val_auc
            counter = 0
                
            save(model.state_dict(), save_best_model_path)
            if show_logs:
                print(f" New best model saved! AUC: {best_val_auc:.4f}")
        else:
            counter += 1
            if show_logs:
                print(f"  No improvement ({counter}/{PATIENCE})")
            if counter >= PATIENCE:
                if show_logs:
                    print("Early Stopping!!")
                break

    # ── RESULTS ────────────────────────────────────────────
    end = time.time()
    hours = (end-start) / 3600
    if show_results:
        model.load_state_dict(load(save_best_model_path, map_location=dev))
        print("Best Model Loaded")

        
        Show_Curves(history)

        all_val_labels, all_val_preds, best_thresholds = model_predict(model, all_splitted_data[1])
        
        all_test_labels, all_test_preds, _ = model_predict(model, all_splitted_data[2])
        
        all_preds_binary = np.zeros_like(all_test_preds, dtype=np.float32)
        for i in range(NUM_OF_CLASSES):
            all_preds_binary[:, i] = (all_test_preds[:, i] > best_thresholds[i]).astype(int)
        
        
        valid_classes = [i for i in range(NUM_OF_CLASSES)
                         if all_test_labels[:, i].sum() > 0]
        
        results_row = {
            'Model'    : name,
            'AUC'      : np.round(roc_auc_score(all_test_labels[:, valid_classes],
                                                 all_test_preds[:, valid_classes],
                                                 average="macro"), 4),
            'F1'       : np.round(f1_score(all_test_labels, all_preds_binary,
                                           average='macro', zero_division=0), 4),
            'Precision': np.round(precision_score(all_test_labels, all_preds_binary,
                                                   average='macro', zero_division=0), 4),
            'Recall'   : np.round(recall_score(all_test_labels, all_preds_binary,
                                               average='macro', zero_division=0), 4),
            "Training_Time (Hours)": np.round(hours, 2),
        }
        
        Show_MCM(all_test_labels, all_preds_binary)  
        
        results_path = r'C:\Users\ibrah.HIMA\OneDrive\Desktop\Full AI\My Reserch Papers\chest-xray-multilabel-classification\results\All_Models_Results.csv'
        results_df = pd.read_csv(results_path) if os.path.exists(results_path) else pd.DataFrame()
        results_df = pd.concat([results_df, pd.DataFrame([results_row])], ignore_index=True)
        results_df.to_csv(results_path, index=False)
        print(results_df)
        
        try:
            save({
                'model_state': model.state_dict(),
                'test_preds' : all_preds_binary,
                'test_probs' : all_test_preds,    
                'test_labels': all_test_labels,   
            }, save_best_model_path)
        except:
            os.remove(save_best_model_path)
            save({
                'model_state': model.state_dict(),
                'test_preds' : all_preds_binary,
                'test_probs' : all_test_preds,   
                'test_labels': all_test_labels,  
            }, save_best_model_path)

In [ ]:
# fit_predict(inception_v3, Inception_V3_Weights,[inc_train_dl,inc_val_dl,inc_test_dl])